# 🌊 Negev Flood Detector — FreeSound Data Collector
Downloads clean, targeted audio clips from FreeSound.org
directly into your Google Drive — fast and precise.

**Output structure:**
```
MyDrive/flood_detector/raw_audio/
├── flood_water/
├── rain/
└── ambient_dry/
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR = '/content/drive/MyDrive/Python course/DIY/final_project'

folders = [
    f'{BASE_DIR}/raw_audio/flood_water',
    f'{BASE_DIR}/raw_audio/rain',
    f'{BASE_DIR}/raw_audio/ambient_dry',
    f'{BASE_DIR}/logs',
]
for folder in folders:
    os.makedirs(folder, exist_ok=True)

print('✅ Drive mounted and folders ready')

## Step 2 — Install & Import

In [ ]:
import requests
import time
import os
import csv
from datetime import datetime

print("✅ All built-in — no extra installs needed")

## Step 3 — Configure API Key & Search Queries
Paste your FreeSound API key below.

In [ ]:
# ============================================================
#  FreeSound API key — DO NOT hardcode secrets in the repo.
#  Get a free key at https://freesound.org/apiv2/apply/
#  Set it as an environment variable, or paste it at the prompt.
# ============================================================
import os, getpass
API_KEY = os.environ.get('FREESOUND_API_KEY') or getpass.getpass('FreeSound API key: ')

MAX_CLIPS_PER_CLASS = 150   # only download what's missing
SLEEP_BETWEEN = 0.5

CLASS_QUERIES = {

    # flood_water — add more diverse water sounds
    'flood_water': [
        'river flooding',
        'flash flood sound',
        'rushing water stream',
        'water torrent',
        'creek rushing water',
        'rapid river flow',
        'flood water rushing',
        'heavy stream water',
        'overflowing river',
        'water surge flood',
    ],

    # rain — already have enough, keep minimal
    'rain': [
        'heavy rain outdoor',
        'rain storm recording',
    ],

    # ambient_dry — THIS IS THE CRITICAL FIX
    # Add indoor silence + room tone so model stops confusing quiet with rain
    'ambient_dry': [
        'room tone silence',
        'indoor ambience quiet',
        'quiet room noise floor',
        'indoor silence recording',
        'office ambience quiet',
        'room silence background',
        'desert wind field recording',
        'dry wind outdoor',
        'arid landscape silence',
        'nature silence no wind',
        'quiet outdoor field recording',
    ],
}

BASE_URL = "https://freesound.org/apiv2"
HEADERS  = {"Authorization": f"Token {API_KEY}"}

resp = requests.get(f"{BASE_URL}/search/text/",
    headers=HEADERS, params={"query": "water", "page_size": 1})

if resp.status_code == 200:
    print("✅ FreeSound connected — API key works!")
elif resp.status_code == 401:
    print("❌ Invalid API key")
else:
    print(f"⚠️ Status: {resp.status_code}")

## Step 4 — Download Clips

In [ ]:
def search_sounds(query, max_results=50):
    sounds = []
    params = {
        "query":     query,
        "page_size": min(max_results, 150),
        "fields":    "id,name,duration,previews",
        "filter":    'duration:[1 TO 30] license:"Creative Commons 0"',
        "sort":      "score",
    }
    try:
        resp = requests.get(f"{BASE_URL}/search/text/",
                            headers=HEADERS, params=params, timeout=15)
        if resp.status_code == 200:
            sounds = resp.json().get("results", [])
        else:
            print(f"     ⚠️ Search error {resp.status_code} for: {query}")
    except Exception as e:
        print(f"     ⚠️ Exception: {e}")
    return sounds


def download_sound(sound, dest_dir):
    try:
        preview_url = sound["previews"]["preview-hq-mp3"]
        safe_name   = sound["name"][:40].replace("/","_").replace(" ","_")
        fname       = f"{sound['id']}_{safe_name}.mp3"
        dest_path   = os.path.join(dest_dir, fname)

        if os.path.exists(dest_path):
            return "skip"

        resp = requests.get(preview_url, headers=HEADERS, timeout=20)
        if resp.status_code == 200:
            with open(dest_path, "wb") as f:
                f.write(resp.content)
            return "ok"
        return "fail"
    except Exception:
        return "fail"


log_rows    = []
grand_total = 0

for class_name, queries in CLASS_QUERIES.items():
    print(f"\n{'='*50}")
    print(f"🔽 [{class_name}]  target: {MAX_CLIPS_PER_CLASS} clips")
    print(f"{'='*50}")

    dest_dir = f"{BASE_DIR}/raw_audio/{class_name}"
    os.makedirs(dest_dir, exist_ok=True)
    seen_ids    = set()
    class_count = 0
    clips_per_query = MAX_CLIPS_PER_CLASS // len(queries) + 10

    for query in queries:
        if class_count >= MAX_CLIPS_PER_CLASS:
            break

        print(f'  🔍 "{query}" ...')
        sounds = search_sounds(query, max_results=clips_per_query)
        print(f"     Found {len(sounds)} results")

        q_ok = 0
        for sound in sounds:
            if class_count >= MAX_CLIPS_PER_CLASS:
                break
            if sound["id"] in seen_ids:
                continue
            seen_ids.add(sound["id"])

            status = download_sound(sound, dest_dir)
            time.sleep(SLEEP_BETWEEN)

            if status == "ok":
                class_count += 1
                q_ok += 1
                log_rows.append({
                    "timestamp": datetime.now().isoformat(),
                    "class":     class_name,
                    "query":     query,
                    "sound_id":  sound["id"],
                    "name":      sound["name"],
                    "duration":  sound["duration"],
                    "status":    "ok",
                })

        print(f"     ✅ {q_ok} downloaded  (class total: {class_count})")

    grand_total += class_count
    print(f"  🏁 [{class_name}] done — {class_count} clips")

log_path = f"{BASE_DIR}/logs/freesound_log.csv"
with open(log_path, "w", newline="") as f:
    writer = csv.DictWriter(f,
        fieldnames=["timestamp","class","query","sound_id","name","duration","status"])
    writer.writeheader()
    writer.writerows(log_rows)

print(f"\n📝 Log saved → {log_path}")
print(f"🎉 Done! {grand_total} total clips downloaded")

## Step 5 — Dataset Summary

In [ ]:
import glob

print('📊 Dataset Summary')
print('=' * 45)
total = 0
for class_name in CLASS_QUERIES.keys():
    class_dir = f'{BASE_DIR}/raw_audio/{class_name}'
    files = glob.glob(f'{class_dir}/*.mp3') + glob.glob(f'{class_dir}/*.wav')
    count = len(files)
    total += count
    bar = '█' * (count // 5)
    print(f'  {class_name:<20} {count:>4} clips  {bar}')
print('=' * 45)
print(f'  TOTAL: {total} clips')

# Drive size
!du -sh {BASE_DIR}/raw_audio/

## Step 6 — Listen to Random Samples

In [ ]:
import glob, random
from IPython.display import Audio, display

for class_name in CLASS_QUERIES.keys():
    files = glob.glob(f'{BASE_DIR}/raw_audio/{class_name}/*.mp3')
    files += glob.glob(f'{BASE_DIR}/raw_audio/{class_name}/*.wav')
    if files:
        sample = random.choice(files)
        print(f'\n🔊 [{class_name}] — {os.path.basename(sample)}')
        display(Audio(sample))
    else:
        print(f'\n⚠️  [{class_name}] — No files found')

## ✅ What's Next?
Data is saved at `MyDrive/flood_detector/raw_audio/`

**Next notebook:** Convert clips → Mel Spectrograms → Train CNN → Export TFLite for UNIHIKER